## 0. Configurando sessão spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_aluno")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

# Projeto usado para faturamento das consultas
spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

:: loading settings :: url = jar:file:/opt/micromamba/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jupyter/.ivy2.5.2/cache
The jars for the packages stored in: /home/jupyter/.ivy2.5.2/jars
com.google.cloud.spark#spark-bigquery-with-dependencies_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-67a465f0-b8c4-43bd-b465-7ccfbdce363a;1.0
	confs: [default]
	found com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 in central
:: resolution report :: resolve 244ms :: artifacts dl 6ms
	:: modules in use:
	com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------------------

In [2]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [3]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [4]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_silver_municipio = f"{par_source_project}.silver.municipio"

par_source_gold_fato_indicador_municipio = f"{par_source_project}.gold.fato_indicador_municipio"

## 3. Leitura dos dados da origem

In [5]:
df_scr_municipio = spark.read.format("bigquery").option("table",par_source_silver_municipio).load()

## 4. Transformações

In [6]:
fato_ind_mun = (
    df_scr_municipio
    .select("ano","id_municipio","rede_id","serie",
            "taxa_alfabetizacao","media_portugues",
            *[f"proporcao_aluno_nivel_{i}" for i in range(9)])
)

## 5. Armazenamento no BQ

In [7]:
(
    fato_ind_mun.write.format("bigquery")
    .option("table", par_source_gold_fato_indicador_municipio)
    .option("writeMethod", "direct")
    .mode("overwrite")
    .save()
)

26/08/26 01:20:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                